In [3]:
from dotenv import load_dotenv
import os
load_dotenv()
from groq import Groq
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [4]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [7]:
assistant = RAGBase(
    index=index,
    llm_client=client,
    instructions=instructions,
)

In [5]:
assistant.rag("How do I run Olama locally?")

'To run Olama locally, there is no information in the provided context.'

In [8]:
messages = [
    {"role": "user", "content": 'I just discovered the course. Can I still join?'},
    ]

In [9]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [10]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"]
        }
    }
}

In [48]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=messages,
    tools=[search_tool],
    tool_choice="auto"
)

In [49]:
len(response.choices)

response.choices

[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=[ChatCompletionMessageToolCall(id='2ysxnh8kb', function=Function(arguments='{"query":"course join requirements"}', name='search'), type='function')]))]

In [51]:
import json

# First check the finish_reason and that tool_calls exists
choice = response.choices[0]
call = choice.message.tool_calls[0]

if choice.finish_reason == "tool_calls" and choice.message.tool_calls:
    tool_call = choice.message.tool_calls[0]
    args = json.loads(tool_call.function.arguments)
    print(args)

{'query': 'course join requirements'}


In [52]:
results = search(**args)

In [53]:
result_json = json.dumps(results, indent=2)

In [54]:
print(result_json) #this is what we're sending back to the llm

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "bd31146b0e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "When will the course be offered next?",
    "answer": "Summer 2025."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certificate if you finish the course with a \"live\" cohort.\n\nWe don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review p

In [55]:
# Add the assistant's response with tool calls
messages.append({
    "role": "assistant",
    "content": None,
    "tool_calls": choice.message.tool_calls
})

In [56]:
# Add the tool result
messages.append({
    "role": "tool",
    "tool_call_id": call.id,
    "content": result_json
})

In [57]:
messages

[{'role': 'user',
  'content': 'I just discovered the course. Can I still join?'},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [ChatCompletionMessageToolCall(id='2ysxnh8kb', function=Function(arguments='{"query":"course join requirements"}', name='search'), type='function')]},
 {'role': 'tool',
  'tool_call_id': '2ysxnh8kb',
  'content': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "bd31146b0e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "When will the course be offered next?",\n    "answer": "Summer 2025."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Quest

In [58]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=messages,
    tools=[search_tool]
)

In [66]:
response

ChatCompletion(id='chatcmpl-8419fc36-a618-4773-b34c-9835f977a5d5', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=None))], created=1781335216, model='llama-3.1-8b-instant', object='chat.completion', mcp_list_tools=None, service_tier='on_demand', system_fingerprint='fp_6a1eabf260', usage=CompletionUsage(completion_tokens=25, prompt_tokens=813, total_tokens=838, completion_time=0.026796944, completion_tokens_details=None, prompt_time=0.051915794, prompt_tokens_details=None, queue_time=0.018402152, total_time=0.078712738), usage_breakdown=None, x_groq=XGroq(id='req_01ktzxmmcnedds7qbnm9e895qa', debug=None, seed=984505985, usage=None))

In [60]:
response.choices[0].message.content.strip()

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [61]:
usage = response.usage
print(usage)

CompletionUsage(completion_tokens=25, prompt_tokens=813, total_tokens=838, completion_time=0.026796944, completion_tokens_details=None, prompt_time=0.051915794, prompt_tokens_details=None, queue_time=0.018402152, total_time=0.078712738)


In [63]:
for item in choice.message.tool_calls:
    print(item)


ChatCompletionMessageToolCall(id='2ysxnh8kb', function=Function(arguments='{"query":"course join requirements"}', name='search'), type='function')


In [11]:
def make_call(call):
    args = json.loads(call.function.arguments)
    
    if call.function.name == "search":
        result = search(**args)
    
    result_json = json.dumps(result, indent=2)
    
    return {
        "role": "tool",
        "tool_call_id": call.id,
        "content": result_json,
    }

In [6]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [12]:
messages = [{"role": "developer", "content": instructions},
            {"role": "user", "content": 'I just discovered the course. Can I still join?'}]

it= 1

while True:
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=messages,
        tools=[search_tool],
    )
    choice = response.choices[0]
    print(choice.finish_reason)

    if choice.finish_reason == "tool_calls":
        messages.append(choice.message)
        for tool_call in choice.message.tool_calls:
            if tool_call.type == 'function':
                print('function_call:', tool_call.function.name, tool_call.function.arguments)
                call_output = make_call(tool_call)
                messages.append(call_output)
        it=it+1
        print(it)

    elif choice.finish_reason == "stop":
        print('ASSISTANT:')
        print(choice.message.content)
        break

stop
ASSISTANT:
<brave_search>Course availability and duration 2023</brave_search>


In [13]:
def agent_loop(instructions, question, model="llama-3.1-8b-instant") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=messages,
            tools=[search_tool],
        )
        choice = response.choices[0]
        print(choice.finish_reason)

        if choice.finish_reason == "tool_calls":
            messages.append(choice.message)
            for tool_call in choice.message.tool_calls:
                if tool_call.type == 'function':
                    print('function_call:', tool_call.function.name, tool_call.function.arguments)
                    call_output = make_call(tool_call)
                    messages.append(call_output)
            it=it+1
            print(it)

        elif choice.finish_reason == "stop":
            print('ASSISTANT:')
            last_answer = choice.message.content.strip()
            break

    return last_answer

In [88]:
agent_loop(instructions, "I just discovered the course. Can I join it?")

tool_calls
function_call: search {"query":"joining the course"}
2
tool_calls
function_call: search {"query":"course joining requirements"}
3
stop
ASSISTANT:


'To join the course, you can start learning and submitting homework while the form is open. However, if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.'

In [90]:
agent_loop(instructions, "what's queen gambit?")

tool_calls
function_call: search {"query":"Queen Gambit"}
2
stop
ASSISTANT:


'It appears I was unable to find any relevant information about a subject or object called "Queen Gambit".'

In [1]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [14]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [15]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [16]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [17]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [18]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

I am working with groq so different syntax etc, ...

In [19]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [ ]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)